# Phase 2 ESN Regression Baseline Tuning

This notebook evaluates a bounded, library-backed Echo State Network regression baseline for the Phase 2 volatility-forecasting task.

Purpose:

- reuse the useful ReservoirPy ESN infrastructure from earlier binary stress-classification work;
- adapt it to continuous realized-volatility targets;
- compare PCA-compressed ESN baselines against the classical HAR/Ridge/ElasticNet floor;
- avoid an extensive hyperparameter campaign.


## 1. Imports and working directory

In [1]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from qpitome_qrc.data.features import FEATURE_COLUMNS, make_sequence_arrays
from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.pca import fit_transform_pca_splits_train_only
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.baselines.esn_regression import (
    ESNRegressionConfig,
    aggregate_esn_regression_seeds,
    fit_single_esn_regressor,
    summarize_esn_regression_run,
)

## 2. Load data and build chronological splits

In [2]:
df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

{name: split.shape for name, split in splits.items()}

{'train': (5459, 41), 'val': (1258, 41), 'test': (1058, 41)}

## 3. Train-only PCA diagnostics

In [3]:
target = "future_rv_20d"
pca6 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=6,
    prefix="pca6",
)
pca6.explained_variance

,component,explained_variance_ratio,cumulative_explained_variance
0,1,0.408573,0.408573
1,2,0.112421,0.520994
2,3,0.090849,0.611843
3,4,0.073259,0.685102
4,5,0.064206,0.749308
5,6,0.055584,0.804892


## 4. Build PCA-6 sequence splits

In [4]:
def build_sequence_splits(transformed_splits, feature_columns, target_column, lookback):
    return {
        name: make_sequence_arrays(
            split,
            feature_columns=feature_columns,
            target_column=target_column,
            lookback=lookback,
        )
        for name, split in transformed_splits.items()
    }

seq_len = 20
sequence_splits = build_sequence_splits(
    pca6.splits,
    pca6.feature_columns,
    target,
    lookback=seq_len,
)

{name: (X.shape, y.shape) for name, (X, y, dates) in sequence_splits.items()}

{'train': ((5440, 20, 6), (5440,)),
 'val': ((1239, 20, 6), (1239,)),
 'test': ((1039, 20, 6), (1039,))}

## 5. Run bounded ESN regression configurations

In [5]:
configs = [
    ESNRegressionConfig(units=300, spectral_radius=0.7, leak_rate=0.5, reservoir_connectivity=0.1, ridge_alpha=1.0, seed=seed)
    for seed in (1, 2, 3)
]

rows = []
for config in configs:
    print(config)
    result = fit_single_esn_regressor(
        sequence_splits,
        config=config,
        target=target,
        feature_set="pca6",
        seq_len=seq_len,
    )
    rows.append(summarize_esn_regression_run(result))

esn_runs = pd.DataFrame(rows)
esn_runs.sort_values("val_rmse")

ESNRegressionConfig(units=300, spectral_radius=0.7, leak_rate=0.5, input_scaling=0.5, input_connectivity=0.5, reservoir_connectivity=0.1, ridge_alpha=1.0, seed=1, washout=0, pooling='final', scale_states=False)
ESNRegressionConfig(units=300, spectral_radius=0.7, leak_rate=0.5, input_scaling=0.5, input_connectivity=0.5, reservoir_connectivity=0.1, ridge_alpha=1.0, seed=2, washout=0, pooling='final', scale_states=False)
ESNRegressionConfig(units=300, spectral_radius=0.7, leak_rate=0.5, input_scaling=0.5, input_connectivity=0.5, reservoir_connectivity=0.1, ridge_alpha=1.0, seed=3, washout=0, pooling='final', scale_states=False)


,units,spectral_radius,leak_rate,input_scaling,input_connectivity,reservoir_connectivity,ridge_alpha,seed,washout,pooling,...,val_rmse,val_qlike,val_mz_alpha,val_mz_beta,val_mz_r2,test_rmse,test_qlike,test_mz_alpha,test_mz_beta,test_mz_r2
1,300,0.7,0.5,0.5,0.5,0.1,1.0,2,0,final,...,0.058897,-3.080875e+00,0.032415,0.635822,0.208156,0.107196,-1.601457,0.046790,0.738697,0.306907
0,300,0.7,0.5,0.5,0.5,0.1,1.0,1,0,final,...,0.059552,7.385001e-01,0.037424,0.605925,0.182060,0.109052,-1.729164,0.051611,0.727758,0.282201
2,300,0.7,0.5,0.5,0.5,0.1,1.0,3,0,final,...,0.059747,2.090111e+06,0.043450,0.570860,0.185717,0.108920,-1.479385,0.046328,0.744727,0.277119


## 6. Aggregate seed stability

In [6]:
esn_aggregate = aggregate_esn_regression_seeds(esn_runs)
esn_aggregate

,model,feature_set,target,seq_len,units,spectral_radius,leak_rate,input_scaling,input_connectivity,reservoir_connectivity,...,scale_states,mean_val_rmse,std_val_rmse,mean_val_qlike,mean_val_mz_r2,mean_test_rmse,std_test_rmse,mean_test_qlike,mean_test_mz_r2,n_seeds
0,esn_regression,pca6,future_rv_20d,20,300,0.7,0.5,0.5,0.5,0.1,...,False,0.059399,0.000445,696703.030488,0.191978,0.10839,0.001035,-1.603335,0.288742,3


## 7. Save exploratory results

In [7]:
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

esn_runs.to_csv(out_dir / "phase2_esn_regression_runs.csv", index=False)
esn_aggregate.to_csv(out_dir / "phase2_esn_regression_seed_aggregate.csv", index=False)
pca6.explained_variance.to_csv(out_dir / "phase2_pca6_explained_variance.csv", index=False)

out_dir

PosixPath('results/tables')

In [8]:
# Log-target ESN regression: same PCA-6 / seq_len=20 / units=300 setup,
# but train Ridge on log(future_rv_20d) and transform predictions back with exp().

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

from qpitome_qrc.baselines.esn_regression import (
    ESNRegressionConfig,
    build_reservoir,
    reservoir_sequence_features,
    scale_sequence_splits,
    maybe_scale_state_features,
)
from qpitome_qrc.evaluation.metrics import evaluate_volatility_forecast

LOG_EPS = 1e-8

configs = [
    ESNRegressionConfig(
        units=300,
        spectral_radius=0.7,
        leak_rate=0.5,
        input_scaling=0.5,
        input_connectivity=0.5,
        reservoir_connectivity=0.1,
        ridge_alpha=1.0,
        seed=seed,
        washout=0,
        pooling="final",
        scale_states=False,
    )
    for seed in (1, 2, 3)
]

rows = []

for config in configs:
    print(config)

    scaled, input_scaler = scale_sequence_splits(sequence_splits)

    X_train, y_train, train_dates = scaled["train"]
    X_val, y_val, val_dates = scaled["val"]
    X_test, y_test, test_dates = scaled["test"]

    reservoir = build_reservoir(config, input_dim=X_train.shape[-1])

    H_train = reservoir_sequence_features(
        reservoir,
        X_train,
        washout=config.washout,
        pooling=config.pooling,
    )
    H_val = reservoir_sequence_features(
        reservoir,
        X_val,
        washout=config.washout,
        pooling=config.pooling,
    )
    H_test = reservoir_sequence_features(
        reservoir,
        X_test,
        washout=config.washout,
        pooling=config.pooling,
    )

    H_train, H_val, H_test, state_scaler = maybe_scale_state_features(
        H_train,
        H_val,
        H_test,
        scale_states=config.scale_states,
    )

    # Train readout on log-volatility.
    y_train_log = np.log(np.maximum(y_train, LOG_EPS))

    readout = Ridge(alpha=config.ridge_alpha)
    readout.fit(H_train, y_train_log)

    # Convert back to original volatility scale.
    train_pred = np.exp(readout.predict(H_train))
    val_pred = np.exp(readout.predict(H_val))
    test_pred = np.exp(readout.predict(H_test))

    train_metrics = evaluate_volatility_forecast(y_train, train_pred)
    val_metrics = evaluate_volatility_forecast(y_val, val_pred)
    test_metrics = evaluate_volatility_forecast(y_test, test_pred)

    rows.append(
        {
            "model": "esn_log_target",
            "feature_set": "pca6",
            "target": target,
            "seq_len": seq_len,
            "units": config.units,
            "spectral_radius": config.spectral_radius,
            "leak_rate": config.leak_rate,
            "input_scaling": config.input_scaling,
            "input_connectivity": config.input_connectivity,
            "reservoir_connectivity": config.reservoir_connectivity,
            "ridge_alpha": config.ridge_alpha,
            "seed": config.seed,
            "washout": config.washout,
            "pooling": config.pooling,
            "scale_states": config.scale_states,
            "train_rmse": train_metrics.rmse,
            "train_qlike": train_metrics.qlike,
            "train_mz_r2": train_metrics.mz_r2,
            "val_rmse": val_metrics.rmse,
            "val_qlike": val_metrics.qlike,
            "val_mz_alpha": val_metrics.mz_alpha,
            "val_mz_beta": val_metrics.mz_beta,
            "val_mz_r2": val_metrics.mz_r2,
            "test_rmse": test_metrics.rmse,
            "test_qlike": test_metrics.qlike,
            "test_mz_alpha": test_metrics.mz_alpha,
            "test_mz_beta": test_metrics.mz_beta,
            "test_mz_r2": test_metrics.mz_r2,
        }
    )

esn_log_runs = pd.DataFrame(rows)
esn_log_runs.sort_values("val_rmse")

ESNRegressionConfig(units=300, spectral_radius=0.7, leak_rate=0.5, input_scaling=0.5, input_connectivity=0.5, reservoir_connectivity=0.1, ridge_alpha=1.0, seed=1, washout=0, pooling='final', scale_states=False)
ESNRegressionConfig(units=300, spectral_radius=0.7, leak_rate=0.5, input_scaling=0.5, input_connectivity=0.5, reservoir_connectivity=0.1, ridge_alpha=1.0, seed=2, washout=0, pooling='final', scale_states=False)
ESNRegressionConfig(units=300, spectral_radius=0.7, leak_rate=0.5, input_scaling=0.5, input_connectivity=0.5, reservoir_connectivity=0.1, ridge_alpha=1.0, seed=3, washout=0, pooling='final', scale_states=False)


,model,feature_set,target,seq_len,units,spectral_radius,leak_rate,input_scaling,input_connectivity,reservoir_connectivity,...,val_rmse,val_qlike,val_mz_alpha,val_mz_beta,val_mz_r2,test_rmse,test_qlike,test_mz_alpha,test_mz_beta,test_mz_r2
0,esn_log_target,pca6,future_rv_20d,20,300,0.7,0.5,0.5,0.5,0.1,...,0.054741,-3.100382,0.019406,0.790140,0.215659,0.109143,-1.719465,0.053105,0.767004,0.277350
1,esn_log_target,pca6,future_rv_20d,20,300,0.7,0.5,0.5,0.5,0.1,...,0.055948,-3.098697,0.028437,0.711838,0.202410,0.117371,-1.638273,0.078862,0.599079,0.228259
2,esn_log_target,pca6,future_rv_20d,20,300,0.7,0.5,0.5,0.5,0.1,...,0.056533,-3.080082,0.036869,0.663551,0.187235,0.109499,-1.667399,0.045699,0.806951,0.260753


In [ ]:
# Small, non-toy ESN log-target regression set based on earlier good settings.
# Assumes existing notebook variables:
#   target = "future_rv_20d"
#   seq_len = 20
#   sequence_splits = PCA-transformed sequence splits


import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

from qpitome_qrc.baselines.esn_regression import (
    ESNRegressionConfig,
    build_reservoir,
    reservoir_sequence_features,
    scale_sequence_splits,
    maybe_scale_state_features,
)
from qpitome_qrc.evaluation.metrics import evaluate_volatility_forecast

LOG_EPS = 1e-8

candidate_configs = [
    (
        "prior_units600_sr0.7_lr0.1_in0.6_alpha1",
        ESNRegressionConfig(
            units=600,
            spectral_radius=0.7,
            leak_rate=0.1,
            input_scaling=0.6,
            input_connectivity=0.5,
            reservoir_connectivity=0.1,
            ridge_alpha=1.0,
            seed=1,
            washout=0,
            pooling="final",
            scale_states=False,
        ),
    ),
    (
        "prior_units600_sr0.7_lr0.1_in0.7_alpha1",
        ESNRegressionConfig(
            units=600,
            spectral_radius=0.7,
            leak_rate=0.1,
            input_scaling=0.7,
            input_connectivity=0.5,
            reservoir_connectivity=0.1,
            ridge_alpha=1.0,
            seed=1,
            washout=0,
            pooling="final",
            scale_states=False,
        ),
    ),
    (
        "prior_units600_sr0.7_lr0.1_in0.6_alpha10",
        ESNRegressionConfig(
            units=600,
            spectral_radius=0.7,
            leak_rate=0.1,
            input_scaling=0.6,
            input_connectivity=0.5,
            reservoir_connectivity=0.1,
            ridge_alpha=10.0,
            seed=1,
            washout=0,
            pooling="final",
            scale_states=False,
        ),
    ),
    (
        "prior_units600_sr0.7_lr0.1_in0.6_alpha0.1",
        ESNRegressionConfig(
            units=600,
            spectral_radius=0.7,
            leak_rate=0.1,
            input_scaling=0.6,
            input_connectivity=0.5,
            reservoir_connectivity=0.1,
            ridge_alpha=0.1,
            seed=1,
            washout=0,
            pooling="final",
            scale_states=False,
        ),
    ),
    (
        "prior_units600_sr0.7_lr0.2_in0.6_alpha1",
        ESNRegressionConfig(
            units=600,
            spectral_radius=0.7,
            leak_rate=0.2,
            input_scaling=0.6,
            input_connectivity=0.5,
            reservoir_connectivity=0.1,
            ridge_alpha=1.0,
            seed=1,
            washout=0,
            pooling="final",
            scale_states=False,
        ),
    ),
    (
        "prior_units600_sr0.9_lr0.1_in0.6_alpha1",
        ESNRegressionConfig(
            units=600,
            spectral_radius=0.9,
            leak_rate=0.1,
            input_scaling=0.6,
            input_connectivity=0.5,
            reservoir_connectivity=0.1,
            ridge_alpha=1.0,
            seed=1,
            washout=0,
            pooling="final",
            scale_states=False,
        ),
    ),
]

rows = []

# Scale sequence inputs once for this feature set / seq_len.
scaled, input_scaler = scale_sequence_splits(sequence_splits)
X_train, y_train, train_dates = scaled["train"]
X_val, y_val, val_dates = scaled["val"]
X_test, y_test, test_dates = scaled["test"]

for label, config in candidate_configs:
    print(label, config)

    reservoir = build_reservoir(config, input_dim=X_train.shape[-1])

    H_train = reservoir_sequence_features(
        reservoir,
        X_train,
        washout=config.washout,
        pooling=config.pooling,
    )
    H_val = reservoir_sequence_features(
        reservoir,
        X_val,
        washout=config.washout,
        pooling=config.pooling,
    )
    H_test = reservoir_sequence_features(
        reservoir,
        X_test,
        washout=config.washout,
        pooling=config.pooling,
    )

    H_train, H_val, H_test, state_scaler = maybe_scale_state_features(
        H_train,
        H_val,
        H_test,
        scale_states=config.scale_states,
    )

    readout = Ridge(alpha=config.ridge_alpha)
    readout.fit(H_train, np.log(np.maximum(y_train, LOG_EPS)))

    train_pred = np.exp(readout.predict(H_train))
    val_pred = np.exp(readout.predict(H_val))
    test_pred = np.exp(readout.predict(H_test))

    train_metrics = evaluate_volatility_forecast(y_train, train_pred)
    val_metrics = evaluate_volatility_forecast(y_val, val_pred)
    test_metrics = evaluate_volatility_forecast(y_test, test_pred)

    rows.append(
        {
            "label": label,
            "model": "esn_log_target",
            "feature_set": "pca6",
            "target": target,
            "seq_len": seq_len,
            "units": config.units,
            "spectral_radius": config.spectral_radius,
            "leak_rate": config.leak_rate,
            "input_scaling": config.input_scaling,
            "input_connectivity": config.input_connectivity,
            "reservoir_connectivity": config.reservoir_connectivity,
            "ridge_alpha": config.ridge_alpha,
            "seed": config.seed,
            "washout": config.washout,
            "pooling": config.pooling,
            "scale_states": config.scale_states,
            "train_rmse": train_metrics.rmse,
            "train_qlike": train_metrics.qlike,
            "train_mz_r2": train_metrics.mz_r2,
            "val_rmse": val_metrics.rmse,
            "val_qlike": val_metrics.qlike,
            "val_mz_alpha": val_metrics.mz_alpha,
            "val_mz_beta": val_metrics.mz_beta,
            "val_mz_r2": val_metrics.mz_r2,
            "test_rmse": test_metrics.rmse,
            "test_qlike": test_metrics.qlike,
            "test_mz_alpha": test_metrics.mz_alpha,
            "test_mz_beta": test_metrics.mz_beta,
            "test_mz_r2": test_metrics.mz_r2,
        }
    )

esn_small_config_test = pd.DataFrame(rows)
esn_small_config_test.sort_values(["val_rmse", "val_qlike"])

prior_units600_sr0.7_lr0.1_in0.6_alpha1 ESNRegressionConfig(units=600, spectral_radius=0.7, leak_rate=0.1, input_scaling=0.6, input_connectivity=0.5, reservoir_connectivity=0.1, ridge_alpha=1.0, seed=1, washout=0, pooling='final', scale_states=False)
prior_units600_sr0.7_lr0.1_in0.7_alpha1 ESNRegressionConfig(units=600, spectral_radius=0.7, leak_rate=0.1, input_scaling=0.7, input_connectivity=0.5, reservoir_connectivity=0.1, ridge_alpha=1.0, seed=1, washout=0, pooling='final', scale_states=False)
prior_units600_sr0.7_lr0.1_in0.6_alpha10 ESNRegressionConfig(units=600, spectral_radius=0.7, leak_rate=0.1, input_scaling=0.6, input_connectivity=0.5, reservoir_connectivity=0.1, ridge_alpha=10.0, seed=1, washout=0, pooling='final', scale_states=False)
prior_units600_sr0.7_lr0.1_in0.6_alpha0.1 ESNRegressionConfig(units=600, spectral_radius=0.7, leak_rate=0.1, input_scaling=0.6, input_connectivity=0.5, reservoir_connectivity=0.1, ridge_alpha=0.1, seed=1, washout=0, pooling='final', scale_states

,label,model,feature_set,target,seq_len,units,spectral_radius,leak_rate,input_scaling,input_connectivity,...,val_rmse,val_qlike,val_mz_alpha,val_mz_beta,val_mz_r2,test_rmse,test_qlike,test_mz_alpha,test_mz_beta,test_mz_r2
2,prior_units600_sr0.7_lr0.1_in0.6_alpha10,esn_log_target,pca6,future_rv_20d,20,600,0.7,0.1,0.6,0.5,...,0.054614,-3.109957,0.019042,0.790978,0.220908,0.119500,-1.613130,0.082467,0.571550,0.210269
0,prior_units600_sr0.7_lr0.1_in0.6_alpha1,esn_log_target,pca6,future_rv_20d,20,600,0.7,0.1,0.6,0.5,...,0.057240,-3.060609,0.035690,0.648445,0.192333,0.133243,-1.259839,0.106781,0.426490,0.161702
1,prior_units600_sr0.7_lr0.1_in0.7_alpha1,esn_log_target,pca6,future_rv_20d,20,600,0.7,0.1,0.7,0.5,...,0.057307,-3.039509,0.035328,0.651248,0.187494,0.134066,-1.035457,0.110285,0.409706,0.134393
5,prior_units600_sr0.9_lr0.1_in0.6_alpha1,esn_log_target,pca6,future_rv_20d,20,600,0.9,0.1,0.6,0.5,...,0.058244,-3.037764,0.040767,0.607654,0.181559,0.130734,-1.340704,0.103110,0.446547,0.165074
4,prior_units600_sr0.7_lr0.2_in0.6_alpha1,esn_log_target,pca6,future_rv_20d,20,600,0.7,0.2,0.6,0.5,...,0.059823,-3.022252,0.048905,0.549278,0.152311,0.128184,-1.288474,0.100086,0.468870,0.162075
3,prior_units600_sr0.7_lr0.1_in0.6_alpha0.1,esn_log_target,pca6,future_rv_20d,20,600,0.7,0.1,0.6,0.5,...,0.063079,-2.836151,0.060968,0.455122,0.136032,0.194850,4.297489,0.152349,0.171938,0.063678


In [10]:
# Next bounded ESN log-target regression test:
# - probe PCA dimension and sequence length
# - keep 300-unit robust config
# - include a few small variations only
#
# Assumes existing notebook variables/functions:
#   df, splits
#   FEATURE_COLUMNS
#   target = "future_rv_20d"
#   fit_transform_pca_splits_train_only
#   make_sequence_arrays
#   ESNRegressionConfig
#   build_reservoir
#   reservoir_sequence_features
#   scale_sequence_splits
#   maybe_scale_state_features
#   evaluate_volatility_forecast

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

LOG_EPS = 1e-8

def build_sequence_splits(transformed_splits, feature_columns, target_column, lookback):
    return {
        name: make_sequence_arrays(
            split,
            feature_columns=feature_columns,
            target_column=target_column,
            lookback=lookback,
        )
        for name, split in transformed_splits.items()
    }


def run_log_target_esn_once(sequence_splits, config, *, feature_set, target, seq_len, label):
    scaled, _ = scale_sequence_splits(sequence_splits)

    X_train, y_train, _ = scaled["train"]
    X_val, y_val, _ = scaled["val"]
    X_test, y_test, _ = scaled["test"]

    reservoir = build_reservoir(config, input_dim=X_train.shape[-1])

    H_train = reservoir_sequence_features(
        reservoir,
        X_train,
        washout=config.washout,
        pooling=config.pooling,
    )
    H_val = reservoir_sequence_features(
        reservoir,
        X_val,
        washout=config.washout,
        pooling=config.pooling,
    )
    H_test = reservoir_sequence_features(
        reservoir,
        X_test,
        washout=config.washout,
        pooling=config.pooling,
    )

    H_train, H_val, H_test, _ = maybe_scale_state_features(
        H_train,
        H_val,
        H_test,
        scale_states=config.scale_states,
    )

    readout = Ridge(alpha=config.ridge_alpha)
    readout.fit(H_train, np.log(np.maximum(y_train, LOG_EPS)))

    train_pred = np.exp(readout.predict(H_train))
    val_pred = np.exp(readout.predict(H_val))
    test_pred = np.exp(readout.predict(H_test))

    train_metrics = evaluate_volatility_forecast(y_train, train_pred)
    val_metrics = evaluate_volatility_forecast(y_val, val_pred)
    test_metrics = evaluate_volatility_forecast(y_test, test_pred)

    return {
        "label": label,
        "model": "esn_log_target",
        "feature_set": feature_set,
        "target": target,
        "seq_len": seq_len,
        "units": config.units,
        "spectral_radius": config.spectral_radius,
        "leak_rate": config.leak_rate,
        "input_scaling": config.input_scaling,
        "input_connectivity": config.input_connectivity,
        "reservoir_connectivity": config.reservoir_connectivity,
        "ridge_alpha": config.ridge_alpha,
        "seed": config.seed,
        "washout": config.washout,
        "pooling": config.pooling,
        "scale_states": config.scale_states,
        "train_rmse": train_metrics.rmse,
        "train_qlike": train_metrics.qlike,
        "train_mz_r2": train_metrics.mz_r2,
        "val_rmse": val_metrics.rmse,
        "val_qlike": val_metrics.qlike,
        "val_mz_alpha": val_metrics.mz_alpha,
        "val_mz_beta": val_metrics.mz_beta,
        "val_mz_r2": val_metrics.mz_r2,
        "test_rmse": test_metrics.rmse,
        "test_qlike": test_metrics.qlike,
        "test_mz_alpha": test_metrics.mz_alpha,
        "test_mz_beta": test_metrics.mz_beta,
        "test_mz_r2": test_metrics.mz_r2,
    }


rows = []
pca_explained_tables = {}

# Bounded search axes.
pca_components_list = [6, 8, 10]
seq_lens = [20, 40]

# Keep this small: configs selected from observed behavior.
base_config_specs = [
    {
        "name": "units300_sr0.7_lr0.5_in0.5_alpha1",
        "units": 300,
        "spectral_radius": 0.7,
        "leak_rate": 0.5,
        "input_scaling": 0.5,
        "ridge_alpha": 1.0,
    },
    {
        "name": "units300_sr0.7_lr0.5_in0.5_alpha10",
        "units": 300,
        "spectral_radius": 0.7,
        "leak_rate": 0.5,
        "input_scaling": 0.5,
        "ridge_alpha": 10.0,
    },
    {
        "name": "units300_sr0.7_lr0.3_in0.5_alpha1",
        "units": 300,
        "spectral_radius": 0.7,
        "leak_rate": 0.3,
        "input_scaling": 0.5,
        "ridge_alpha": 1.0,
    },
    {
        "name": "units300_sr0.9_lr0.5_in0.5_alpha1",
        "units": 300,
        "spectral_radius": 0.9,
        "leak_rate": 0.5,
        "input_scaling": 0.5,
        "ridge_alpha": 1.0,
    },
]

seeds = [1, 2, 3]

for n_components in pca_components_list:
    feature_set_name = f"pca{n_components}"

    pca_result = fit_transform_pca_splits_train_only(
        splits,
        feature_columns=FEATURE_COLUMNS,
        target_columns=[target],
        n_components=n_components,
        prefix=feature_set_name,
    )
    pca_explained_tables[feature_set_name] = pca_result.explained_variance

    for seq_len_i in seq_lens:
        print(f"\n=== {feature_set_name}, seq_len={seq_len_i} ===")

        sequence_splits_i = build_sequence_splits(
            pca_result.splits,
            pca_result.feature_columns,
            target,
            lookback=seq_len_i,
        )

        for spec in base_config_specs:
            for seed in seeds:
                config = ESNRegressionConfig(
                    units=spec["units"],
                    spectral_radius=spec["spectral_radius"],
                    leak_rate=spec["leak_rate"],
                    input_scaling=spec["input_scaling"],
                    input_connectivity=0.5,
                    reservoir_connectivity=0.1,
                    ridge_alpha=spec["ridge_alpha"],
                    seed=seed,
                    washout=0,
                    pooling="final",
                    scale_states=False,
                )

                label = f"{feature_set_name}_seq{seq_len_i}_{spec['name']}_seed{seed}"
                print(label)

                rows.append(
                    run_log_target_esn_once(
                        sequence_splits_i,
                        config,
                        feature_set=feature_set_name,
                        target=target,
                        seq_len=seq_len_i,
                        label=label,
                    )
                )

esn_next_test = pd.DataFrame(rows)

config_cols = [
    "feature_set",
    "target",
    "seq_len",
    "units",
    "spectral_radius",
    "leak_rate",
    "input_scaling",
    "input_connectivity",
    "reservoir_connectivity",
    "ridge_alpha",
    "washout",
    "pooling",
    "scale_states",
]

esn_next_aggregate = (
    esn_next_test
    .groupby(config_cols, as_index=False)
    .agg(
        mean_val_rmse=("val_rmse", "mean"),
        std_val_rmse=("val_rmse", "std"),
        mean_val_qlike=("val_qlike", "mean"),
        mean_val_mz_r2=("val_mz_r2", "mean"),
        mean_test_rmse=("test_rmse", "mean"),
        std_test_rmse=("test_rmse", "std"),
        mean_test_qlike=("test_qlike", "mean"),
        mean_test_mz_r2=("test_mz_r2", "mean"),
        n_seeds=("seed", "size"),
    )
    .sort_values(["mean_val_rmse", "mean_val_qlike"])
    .reset_index(drop=True)
)

display(esn_next_aggregate.head(20))
display(esn_next_test.sort_values(["val_rmse", "val_qlike"]).head(20))


=== pca6, seq_len=20 ===
pca6_seq20_units300_sr0.7_lr0.5_in0.5_alpha1_seed1
pca6_seq20_units300_sr0.7_lr0.5_in0.5_alpha1_seed2
pca6_seq20_units300_sr0.7_lr0.5_in0.5_alpha1_seed3
pca6_seq20_units300_sr0.7_lr0.5_in0.5_alpha10_seed1
pca6_seq20_units300_sr0.7_lr0.5_in0.5_alpha10_seed2
pca6_seq20_units300_sr0.7_lr0.5_in0.5_alpha10_seed3
pca6_seq20_units300_sr0.7_lr0.3_in0.5_alpha1_seed1
pca6_seq20_units300_sr0.7_lr0.3_in0.5_alpha1_seed2
pca6_seq20_units300_sr0.7_lr0.3_in0.5_alpha1_seed3
pca6_seq20_units300_sr0.9_lr0.5_in0.5_alpha1_seed1
pca6_seq20_units300_sr0.9_lr0.5_in0.5_alpha1_seed2
pca6_seq20_units300_sr0.9_lr0.5_in0.5_alpha1_seed3

=== pca6, seq_len=40 ===
pca6_seq40_units300_sr0.7_lr0.5_in0.5_alpha1_seed1
pca6_seq40_units300_sr0.7_lr0.5_in0.5_alpha1_seed2
pca6_seq40_units300_sr0.7_lr0.5_in0.5_alpha1_seed3
pca6_seq40_units300_sr0.7_lr0.5_in0.5_alpha10_seed1
pca6_seq40_units300_sr0.7_lr0.5_in0.5_alpha10_seed2
pca6_seq40_units300_sr0.7_lr0.5_in0.5_alpha10_seed3
pca6_seq40_units300_sr0.

,feature_set,target,seq_len,units,spectral_radius,leak_rate,input_scaling,input_connectivity,reservoir_connectivity,ridge_alpha,...,scale_states,mean_val_rmse,std_val_rmse,mean_val_qlike,mean_val_mz_r2,mean_test_rmse,std_test_rmse,mean_test_qlike,mean_test_mz_r2,n_seeds
0,pca8,future_rv_20d,20,300,0.7,0.5,0.5,0.5,0.1,10.0,...,False,0.052868,0.000474,-3.111317,0.248808,0.111961,0.005436,-1.725294,0.294252,3
1,pca10,future_rv_20d,20,300,0.7,0.5,0.5,0.5,0.1,10.0,...,False,0.053023,0.000144,-3.106449,0.244875,0.108556,0.000747,-1.679948,0.283820,3
2,pca8,future_rv_20d,40,300,0.7,0.5,0.5,0.5,0.1,10.0,...,False,0.053119,0.000505,-3.108984,0.252052,0.081531,0.006150,-2.475993,0.487586,3
3,pca10,future_rv_20d,40,300,0.7,0.5,0.5,0.5,0.1,10.0,...,False,0.053278,0.000150,-3.104190,0.247859,0.076335,0.000930,-2.474226,0.485579,3
4,pca6,future_rv_20d,20,300,0.7,0.5,0.5,0.5,0.1,10.0,...,False,0.053584,0.000506,-3.113025,0.237359,0.107712,0.000849,-1.782864,0.289445,3
5,pca10,future_rv_20d,20,300,0.7,0.5,0.5,0.5,0.1,1.0,...,False,0.053796,0.000077,-3.101885,0.231125,0.111234,0.001039,-1.585105,0.263475,3
6,pca10,future_rv_20d,20,300,0.9,0.5,0.5,0.5,0.1,1.0,...,False,0.053835,0.000664,-3.098441,0.229803,0.111672,0.000819,-1.607981,0.261963,3
7,pca6,future_rv_20d,40,300,0.7,0.5,0.5,0.5,0.1,10.0,...,False,0.053838,0.000503,-3.110378,0.240443,0.076365,0.001339,-2.483579,0.484284,3
8,pca10,future_rv_20d,40,300,0.7,0.5,0.5,0.5,0.1,1.0,...,False,0.054046,0.000061,-3.099795,0.234136,0.079509,0.000674,-2.460160,0.458009,3
9,pca10,future_rv_20d,40,300,0.9,0.5,0.5,0.5,0.1,1.0,...,False,0.054074,0.000692,-3.096864,0.233150,0.080353,0.001306,-2.457776,0.451917,3


,label,model,feature_set,target,seq_len,units,spectral_radius,leak_rate,input_scaling,input_connectivity,...,val_rmse,val_qlike,val_mz_alpha,val_mz_beta,val_mz_r2,test_rmse,test_qlike,test_mz_alpha,test_mz_beta,test_mz_r2
29,pca8_seq20_units300_sr0.7_lr0.5_in0.5_alpha10_...,esn_log_target,pca8,future_rv_20d,20,300,0.7,0.5,0.5,0.5,...,0.052479,-3.112101,0.007686,0.929992,0.254120,0.115734,-1.707757,0.080300,0.598830,0.277274
41,pca8_seq40_units300_sr0.7_lr0.5_in0.5_alpha10_...,esn_log_target,pca8,future_rv_20d,40,300,0.7,0.5,0.5,0.5,...,0.052713,-3.110057,0.006667,0.938275,0.257895,0.086240,-2.475068,0.062989,0.643929,0.464695
27,pca8_seq20_units300_sr0.7_lr0.5_in0.5_alpha10_...,esn_log_target,pca8,future_rv_20d,20,300,0.7,0.5,0.5,0.5,...,0.052730,-3.109710,0.017497,0.849020,0.253485,0.114417,-1.643465,0.076520,0.620508,0.275130
52,pca10_seq20_units300_sr0.7_lr0.5_in0.5_alpha10...,esn_log_target,pca10,future_rv_20d,20,300,0.7,0.5,0.5,0.5,...,0.052920,-3.110442,0.017243,0.844667,0.248985,0.107699,-1.677714,0.041700,0.837984,0.284087
39,pca8_seq40_units300_sr0.7_lr0.5_in0.5_alpha10_...,esn_log_target,pca8,future_rv_20d,40,300,0.7,0.5,0.5,0.5,...,0.052959,-3.107672,0.016686,0.855861,0.257079,0.083780,-2.465106,0.057645,0.673763,0.471183
53,pca10_seq20_units300_sr0.7_lr0.5_in0.5_alpha10...,esn_log_target,pca10,future_rv_20d,20,300,0.7,0.5,0.5,0.5,...,0.052961,-3.106547,0.011735,0.887663,0.243592,0.108899,-1.671833,0.047382,0.817243,0.275695
32,pca8_seq20_units300_sr0.7_lr0.3_in0.5_alpha1_s...,esn_log_target,pca8,future_rv_20d,20,300,0.7,0.3,0.5,0.5,...,0.053034,-3.111045,0.020145,0.814639,0.251241,0.151067,-1.530985,0.123476,0.354161,0.192337
57,pca10_seq20_units300_sr0.9_lr0.5_in0.5_alpha1_...,esn_log_target,pca10,future_rv_20d,20,300,0.9,0.5,0.5,0.5,...,0.053069,-3.104644,0.021979,0.812951,0.248878,0.112252,-1.615420,0.070021,0.660245,0.278082
3,pca6_seq20_units300_sr0.7_lr0.5_in0.5_alpha10_...,esn_log_target,pca6,future_rv_20d,20,300,0.7,0.5,0.5,0.5,...,0.053108,-3.116032,0.010133,0.875255,0.247510,0.107232,-1.803984,0.049251,0.781692,0.298849
51,pca10_seq20_units300_sr0.7_lr0.5_in0.5_alpha10...,esn_log_target,pca10,future_rv_20d,20,300,0.7,0.5,0.5,0.5,...,0.053188,-3.102356,0.018894,0.834315,0.242048,0.109070,-1.690298,0.058956,0.726529,0.291679


In [11]:
# Focused confirmation run:
#   PCA-6 / PCA-8 / PCA-10
#   seq_len = 40
#   units = 300
#   sr = 0.7
#   leak = 0.5
#   input_scaling = 0.5
#   ridge_alpha = 10
#   seeds = 1..5
#
# Assumes existing notebook variables/imports from previous cells:
#   splits, FEATURE_COLUMNS, target
#   fit_transform_pca_splits_train_only
#   make_sequence_arrays
#   ESNRegressionConfig
#   build_reservoir
#   reservoir_sequence_features
#   scale_sequence_splits
#   maybe_scale_state_features
#   evaluate_volatility_forecast

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

LOG_EPS = 1e-8


def build_sequence_splits(transformed_splits, feature_columns, target_column, lookback):
    return {
        name: make_sequence_arrays(
            split,
            feature_columns=feature_columns,
            target_column=target_column,
            lookback=lookback,
        )
        for name, split in transformed_splits.items()
    }


def run_log_target_esn_once(sequence_splits, config, *, feature_set, target, seq_len, label):
    scaled, _ = scale_sequence_splits(sequence_splits)

    X_train, y_train, _ = scaled["train"]
    X_val, y_val, _ = scaled["val"]
    X_test, y_test, _ = scaled["test"]

    reservoir = build_reservoir(config, input_dim=X_train.shape[-1])

    H_train = reservoir_sequence_features(
        reservoir,
        X_train,
        washout=config.washout,
        pooling=config.pooling,
    )
    H_val = reservoir_sequence_features(
        reservoir,
        X_val,
        washout=config.washout,
        pooling=config.pooling,
    )
    H_test = reservoir_sequence_features(
        reservoir,
        X_test,
        washout=config.washout,
        pooling=config.pooling,
    )

    H_train, H_val, H_test, _ = maybe_scale_state_features(
        H_train,
        H_val,
        H_test,
        scale_states=config.scale_states,
    )

    readout = Ridge(alpha=config.ridge_alpha)
    readout.fit(H_train, np.log(np.maximum(y_train, LOG_EPS)))

    train_pred = np.exp(readout.predict(H_train))
    val_pred = np.exp(readout.predict(H_val))
    test_pred = np.exp(readout.predict(H_test))

    train_metrics = evaluate_volatility_forecast(y_train, train_pred)
    val_metrics = evaluate_volatility_forecast(y_val, val_pred)
    test_metrics = evaluate_volatility_forecast(y_test, test_pred)

    return {
        "label": label,
        "model": "esn_log_target",
        "feature_set": feature_set,
        "target": target,
        "seq_len": seq_len,
        "units": config.units,
        "spectral_radius": config.spectral_radius,
        "leak_rate": config.leak_rate,
        "input_scaling": config.input_scaling,
        "input_connectivity": config.input_connectivity,
        "reservoir_connectivity": config.reservoir_connectivity,
        "ridge_alpha": config.ridge_alpha,
        "seed": config.seed,
        "washout": config.washout,
        "pooling": config.pooling,
        "scale_states": config.scale_states,
        "train_rmse": train_metrics.rmse,
        "train_qlike": train_metrics.qlike,
        "train_mz_alpha": train_metrics.mz_alpha,
        "train_mz_beta": train_metrics.mz_beta,
        "train_mz_r2": train_metrics.mz_r2,
        "val_rmse": val_metrics.rmse,
        "val_qlike": val_metrics.qlike,
        "val_mz_alpha": val_metrics.mz_alpha,
        "val_mz_beta": val_metrics.mz_beta,
        "val_mz_r2": val_metrics.mz_r2,
        "test_rmse": test_metrics.rmse,
        "test_qlike": test_metrics.qlike,
        "test_mz_alpha": test_metrics.mz_alpha,
        "test_mz_beta": test_metrics.mz_beta,
        "test_mz_r2": test_metrics.mz_r2,
    }


rows = []
pca_explained_tables = {}

confirmation_settings = {
    "pca_components": [6, 8, 10],
    "seq_len": 40,
    "seeds": [1, 2, 3, 4, 5],
    "config": {
        "units": 300,
        "spectral_radius": 0.7,
        "leak_rate": 0.5,
        "input_scaling": 0.5,
        "input_connectivity": 0.5,
        "reservoir_connectivity": 0.1,
        "ridge_alpha": 10.0,
        "washout": 0,
        "pooling": "final",
        "scale_states": False,
    },
}

for n_components in confirmation_settings["pca_components"]:
    feature_set_name = f"pca{n_components}"
    seq_len = confirmation_settings["seq_len"]

    print(f"\n=== {feature_set_name}, seq_len={seq_len} ===")

    pca_result = fit_transform_pca_splits_train_only(
        splits,
        feature_columns=FEATURE_COLUMNS,
        target_columns=[target],
        n_components=n_components,
        prefix=feature_set_name,
    )
    pca_explained_tables[feature_set_name] = pca_result.explained_variance

    sequence_splits_i = build_sequence_splits(
        pca_result.splits,
        pca_result.feature_columns,
        target,
        lookback=seq_len,
    )

    for seed in confirmation_settings["seeds"]:
        config = ESNRegressionConfig(
            seed=seed,
            **confirmation_settings["config"],
        )

        label = (
            f"{feature_set_name}_seq{seq_len}_"
            f"units{config.units}_sr{config.spectral_radius}_"
            f"lr{config.leak_rate}_in{config.input_scaling}_"
            f"alpha{config.ridge_alpha}_seed{seed}"
        )

        print(label)

        rows.append(
            run_log_target_esn_once(
                sequence_splits_i,
                config,
                feature_set=feature_set_name,
                target=target,
                seq_len=seq_len,
                label=label,
            )
        )

esn_confirmation_runs = pd.DataFrame(rows)

config_cols = [
    "feature_set",
    "target",
    "seq_len",
    "units",
    "spectral_radius",
    "leak_rate",
    "input_scaling",
    "input_connectivity",
    "reservoir_connectivity",
    "ridge_alpha",
    "washout",
    "pooling",
    "scale_states",
]

esn_confirmation_aggregate = (
    esn_confirmation_runs
    .groupby(config_cols, as_index=False)
    .agg(
        mean_val_rmse=("val_rmse", "mean"),
        std_val_rmse=("val_rmse", "std"),
        mean_val_qlike=("val_qlike", "mean"),
        std_val_qlike=("val_qlike", "std"),
        mean_val_mz_r2=("val_mz_r2", "mean"),
        std_val_mz_r2=("val_mz_r2", "std"),
        mean_test_rmse=("test_rmse", "mean"),
        std_test_rmse=("test_rmse", "std"),
        mean_test_qlike=("test_qlike", "mean"),
        std_test_qlike=("test_qlike", "std"),
        mean_test_mz_r2=("test_mz_r2", "mean"),
        std_test_mz_r2=("test_mz_r2", "std"),
        n_seeds=("seed", "size"),
    )
    .sort_values(["mean_val_rmse", "mean_val_qlike"])
    .reset_index(drop=True)
)

display(esn_confirmation_aggregate)
display(esn_confirmation_runs.sort_values(["val_rmse", "val_qlike"]).head(20))

# Optional save
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

esn_confirmation_runs.to_csv(
    out_dir / "phase2_esn_log_target_confirmation_runs.csv",
    index=False,
)
esn_confirmation_aggregate.to_csv(
    out_dir / "phase2_esn_log_target_confirmation_aggregate.csv",
    index=False,
)

for name, table in pca_explained_tables.items():
    table.to_csv(out_dir / f"phase2_{name}_explained_variance.csv", index=False)

print("Saved ESN confirmation outputs to:", out_dir)


=== pca6, seq_len=40 ===
pca6_seq40_units300_sr0.7_lr0.5_in0.5_alpha10.0_seed1
pca6_seq40_units300_sr0.7_lr0.5_in0.5_alpha10.0_seed2
pca6_seq40_units300_sr0.7_lr0.5_in0.5_alpha10.0_seed3
pca6_seq40_units300_sr0.7_lr0.5_in0.5_alpha10.0_seed4
pca6_seq40_units300_sr0.7_lr0.5_in0.5_alpha10.0_seed5

=== pca8, seq_len=40 ===
pca8_seq40_units300_sr0.7_lr0.5_in0.5_alpha10.0_seed1
pca8_seq40_units300_sr0.7_lr0.5_in0.5_alpha10.0_seed2
pca8_seq40_units300_sr0.7_lr0.5_in0.5_alpha10.0_seed3
pca8_seq40_units300_sr0.7_lr0.5_in0.5_alpha10.0_seed4
pca8_seq40_units300_sr0.7_lr0.5_in0.5_alpha10.0_seed5

=== pca10, seq_len=40 ===
pca10_seq40_units300_sr0.7_lr0.5_in0.5_alpha10.0_seed1
pca10_seq40_units300_sr0.7_lr0.5_in0.5_alpha10.0_seed2
pca10_seq40_units300_sr0.7_lr0.5_in0.5_alpha10.0_seed3
pca10_seq40_units300_sr0.7_lr0.5_in0.5_alpha10.0_seed4
pca10_seq40_units300_sr0.7_lr0.5_in0.5_alpha10.0_seed5


,feature_set,target,seq_len,units,spectral_radius,leak_rate,input_scaling,input_connectivity,reservoir_connectivity,ridge_alpha,...,std_val_qlike,mean_val_mz_r2,std_val_mz_r2,mean_test_rmse,std_test_rmse,mean_test_qlike,std_test_qlike,mean_test_mz_r2,std_test_mz_r2,n_seeds
0,pca8,future_rv_20d,40,300,0.7,0.5,0.5,0.5,0.1,10.0,...,0.004268,0.252864,0.006775,0.080012,0.005319,-2.467632,0.017073,0.487107,0.037548,5
1,pca10,future_rv_20d,40,300,0.7,0.5,0.5,0.5,0.1,10.0,...,0.004307,0.247563,0.006114,0.077039,0.001885,-2.464078,0.016850,0.478129,0.025629,5
2,pca6,future_rv_20d,40,300,0.7,0.5,0.5,0.5,0.1,10.0,...,0.005452,0.239929,0.008458,0.077063,0.002838,-2.481683,0.003812,0.485518,0.023783,5


,label,model,feature_set,target,seq_len,units,spectral_radius,leak_rate,input_scaling,input_connectivity,...,val_rmse,val_qlike,val_mz_alpha,val_mz_beta,val_mz_r2,test_rmse,test_qlike,test_mz_alpha,test_mz_beta,test_mz_r2
7,pca8_seq40_units300_sr0.7_lr0.5_in0.5_alpha10....,esn_log_target,pca8,future_rv_20d,40,300,0.7,0.5,0.5,0.5,...,0.052713,-3.110057,0.006667,0.938275,0.257895,0.086240,-2.475068,0.062989,0.643929,0.464695
14,pca10_seq40_units300_sr0.7_lr0.5_in0.5_alpha10...,esn_log_target,pca10,future_rv_20d,40,300,0.7,0.5,0.5,0.5,...,0.052910,-3.096660,0.013352,0.889339,0.255024,0.080189,-2.438010,0.035933,0.827007,0.437189
5,pca8_seq40_units300_sr0.7_lr0.5_in0.5_alpha10....,esn_log_target,pca8,future_rv_20d,40,300,0.7,0.5,0.5,0.5,...,0.052959,-3.107672,0.016686,0.855861,0.257079,0.083780,-2.465106,0.057645,0.673763,0.471183
8,pca8_seq40_units300_sr0.7_lr0.5_in0.5_alpha10....,esn_log_target,pca8,future_rv_20d,40,300,0.7,0.5,0.5,0.5,...,0.052975,-3.118570,0.011729,0.892336,0.253349,0.074554,-2.468924,0.038070,0.802530,0.527013
9,pca8_seq40_units300_sr0.7_lr0.5_in0.5_alpha10....,esn_log_target,pca8,future_rv_20d,40,300,0.7,0.5,0.5,0.5,...,0.053051,-3.110254,0.013262,0.869192,0.254816,0.080914,-2.441259,0.045697,0.782989,0.445765
11,pca10_seq40_units300_sr0.7_lr0.5_in0.5_alpha10...,esn_log_target,pca10,future_rv_20d,40,300,0.7,0.5,0.5,0.5,...,0.053189,-3.107661,0.016280,0.851511,0.251572,0.075308,-2.482319,0.018190,0.919491,0.488869
12,pca10_seq40_units300_sr0.7_lr0.5_in0.5_alpha10...,esn_log_target,pca10,future_rv_20d,40,300,0.7,0.5,0.5,0.5,...,0.053193,-3.105006,0.010473,0.897545,0.247278,0.077121,-2.466313,0.025207,0.891917,0.469392
0,pca6_seq40_units300_sr0.7_lr0.5_in0.5_alpha10....,esn_log_target,pca6,future_rv_20d,40,300,0.7,0.5,0.5,0.5,...,0.053383,-3.113030,0.009619,0.879604,0.250107,0.075499,-2.486982,0.029426,0.841454,0.498777
13,pca10_seq40_units300_sr0.7_lr0.5_in0.5_alpha10...,esn_log_target,pca10,future_rv_20d,40,300,0.7,0.5,0.5,0.5,...,0.053440,-3.103160,0.010704,0.901409,0.239216,0.076000,-2.459702,0.034366,0.844835,0.496718
10,pca10_seq40_units300_sr0.7_lr0.5_in0.5_alpha10...,esn_log_target,pca10,future_rv_20d,40,300,0.7,0.5,0.5,0.5,...,0.053452,-3.099903,0.017972,0.841266,0.244727,0.076576,-2.474045,0.037277,0.797067,0.498477


Saved ESN confirmation outputs to: results/tables
